# Feature Selection — Problem B (`delivery_days`)

Changes from the first version:

1. The correlation filter is now applied before Lasso (it was computed but never used).
2. Features are chosen by **stability selection**: Lasso at the CV-best alpha, refitted on every time-ordered fold; a feature is kept if it is selected in at least 80% of folds (previously any non-zero one-hot dummy kept the whole column, so 57 of 65 features survived).
3. Cross-validation uses `TimeSeriesSplit` on time-sorted training rows, so every fold trains on earlier orders only.
4. The target is `log(delivery_days / promised_delivery_days)`, which removes the drift in delivery time between train, validation and test.
5. Every model is compared with a naive baseline: `promised_delivery_days` × the median train ratio.

## Load dataset

In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, LassoCV, LinearRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.append(str(ROOT))

from src.config import CV_FOLDS, TABLES as TABLES_DIR, set_seed
from src.features import load_feature_table

set_seed()

In [2]:
# Cleaned regression file + Part A and Part B features (run 02_preprocessing first)
df = load_feature_table("regression")

df["order_purchase_timestamp"] = pd.to_datetime(
    df["order_purchase_timestamp"]
)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (96454, 72)

Columns:
['order_id', 'order_purchase_timestamp', 'purchase_month', 'customer_state', 'customer_region', 'customer_zip_code_prefix', 'customer_zip_lat', 'customer_zip_lng', 'customer_zip_density', 'customer_is_capital', 'remote_flag', 'order_value', 'mean_item_price', 'max_item_price', 'freight_value', 'item_count', 'unique_product_count', 'seller_count', 'seller_state_count', 'product_category_count', 'multi_seller_flag', 'product_category', 'primary_seller_state', 'primary_seller_region', 'region_pair', 'same_state', 'any_seller_same_state', 'same_city', 'any_seller_same_city', 'mean_seller_zip_lat', 'mean_seller_zip_lng', 'estimated_min_distance_km', 'estimated_mean_distance_km', 'estimated_max_distance_km', 'geolocation_missing', 'total_weight_g', 'total_volume_cm3', 'max_item_weight_g', 'longest_side_cm', 'max_sum_of_dims_cm', 'volumetric_weight_kg', 'billable_weight_kg', 'density_g_cm3', 'payment_value', 'payment_installments', 'payment_type_count', 'p

## 1. Remove ID / Target / Split / Obvious duplicate variables

In [3]:
target = "delivery_days"

# exclude target and non-feature columns
exclude_cols = [
    "order_id",                    # ID
    "order_purchase_timestamp",    # 已经生成时间衍生变量
    "split",                       # Train/Val/Test标签
    "delivery_days"                # Target
]

# exclude duplicate month features
# keep
# purchase_month_sin
# purchase_month_cos
duplicate_cols = [
    "purchase_month",
    "purchase_month_of_year"
]

# exclude customer_zip_code_prefix
manual_drop_cols = exclude_cols + duplicate_cols + [
    "customer_zip_code_prefix"
]

# features after manual drop
feature_cols = [
    col for col in df.columns
    if col not in manual_drop_cols
]

print("Number of features after Step 1:", len(feature_cols))
print(feature_cols)

Number of features after Step 1: 65
['customer_state', 'customer_region', 'customer_zip_lat', 'customer_zip_lng', 'customer_zip_density', 'customer_is_capital', 'remote_flag', 'order_value', 'mean_item_price', 'max_item_price', 'freight_value', 'item_count', 'unique_product_count', 'seller_count', 'seller_state_count', 'product_category_count', 'multi_seller_flag', 'product_category', 'primary_seller_state', 'primary_seller_region', 'region_pair', 'same_state', 'any_seller_same_state', 'same_city', 'any_seller_same_city', 'mean_seller_zip_lat', 'mean_seller_zip_lng', 'estimated_min_distance_km', 'estimated_mean_distance_km', 'estimated_max_distance_km', 'geolocation_missing', 'total_weight_g', 'total_volume_cm3', 'max_item_weight_g', 'longest_side_cm', 'max_sum_of_dims_cm', 'volumetric_weight_kg', 'billable_weight_kg', 'density_g_cm3', 'payment_value', 'payment_installments', 'payment_type_count', 'payment_type', 'used_voucher', 'payment_missing', 'promised_delivery_days', 'promised_da

## 2. Remove redundant variables with a correlation greater than 0.90.

In [4]:
# Split the dataset into train, validation, and test sets
# Sort by purchase time first: TimeSeriesSplit assumes rows are in time order,
# so each CV fold trains on earlier orders and validates on later ones.
df = df.sort_values("order_purchase_timestamp").reset_index(drop=True)

train_df = df[df["split"] == "train"].copy()

val_df = df[
    df["split"].isin(["validation", "val"])
].copy()

test_df = df[df["split"] == "test"].copy()

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

X_train = train_df[feature_cols].copy()
y_train = train_df[target].copy()

X_val = val_df[feature_cols].copy()
y_val = val_df[target].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df[target].copy()

Train: (83947, 72)
Validation: (6156, 72)
Test: (6351, 72)


In [5]:
# check correlation > 0.90
# 只看 Train 中的数值变量

numeric_cols = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

print("Number of numeric features:", len(numeric_cols))

corr_matrix = (
    X_train[numeric_cols]
    .corr()
    .abs()
)

upper = corr_matrix.where(
    np.triu(
        np.ones(corr_matrix.shape),
        k=1
    ).astype(bool)
)

high_corr_pairs = (
    upper
    .stack()
    .sort_values(ascending=False)
)

high_corr_pairs = high_corr_pairs[
    high_corr_pairs > 0.90
]

print("Highly correlated pairs (> 0.90):")
print(high_corr_pairs)

Number of numeric features: 58


Highly correlated pairs (> 0.90):
total_volume_cm3            volumetric_weight_kg          1.000000
estimated_min_distance_km   estimated_mean_distance_km    0.999548
estimated_mean_distance_km  estimated_max_distance_km     0.999546
estimated_min_distance_km   estimated_max_distance_km     0.998192
mean_item_price             max_item_price                0.998121
seller_past_orders          seller_past_deliveries        0.997864
order_value                 payment_value                 0.996257
same_state                  any_seller_same_state         0.994763
same_city                   any_seller_same_city          0.993858
seller_count                multi_seller_flag             0.973525
promised_delivery_days      weekend_days_to_estimate      0.970760
total_volume_cm3            billable_weight_kg            0.948528
volumetric_weight_kg        billable_weight_kg            0.948528
order_value                 max_item_price                0.935482
                            

In [6]:
# mannually drop one of each pair of highly correlated features
corr_manual_drop = [
    "volumetric_weight_kg",

    "estimated_min_distance_km",
    "estimated_mean_distance_km",

    "max_item_price",
    "mean_item_price",          # 0.93 with order_value

    "seller_past_deliveries",

    "payment_value",

    "same_state",
    "same_city",

    "multi_seller_flag",

    "weekend_days_to_estimate",

    "max_sum_of_dims_cm",

    "billable_weight_kg"        # 0.95 with total_volume_cm3, 0.93 with total_weight_g
]

feature_cols_corr = [
    col for col in feature_cols
    if col not in corr_manual_drop
]

# Apply the filter: every later step uses these columns only
X_train = X_train[feature_cols_corr]
X_val = X_val[feature_cols_corr]
X_test = X_test[feature_cols_corr]

remaining_corr = (
    X_train
    .select_dtypes(include=["number"])
    .corr()
    .abs()
)
remaining_corr = remaining_corr.where(
    np.triu(np.ones(remaining_corr.shape), k=1).astype(bool)
).stack()
assert (remaining_corr <= 0.90).all(), remaining_corr[remaining_corr > 0.90]

print("Features before correlation filtering:", len(feature_cols))
print("Features after correlation filtering:", len(feature_cols_corr))
print("Highest remaining correlation:", round(remaining_corr.max(), 3))

print("\nDropped features:")
print(corr_manual_drop)

Features before correlation filtering: 65
Features after correlation filtering: 52
Highest remaining correlation: 0.888

Dropped features:
['volumetric_weight_kg', 'estimated_min_distance_km', 'estimated_mean_distance_km', 'max_item_price', 'mean_item_price', 'seller_past_deliveries', 'payment_value', 'same_state', 'same_city', 'multi_seller_flag', 'weekend_days_to_estimate', 'max_sum_of_dims_cm', 'billable_weight_kg']


## 3. Target and baseline

Delivery time falls sharply over time, so a model fitted on the raw target over-predicts later months.
Olist's own promise (`promised_delivery_days`) falls with it, so the model predicts the delivery time **relative to the promise**:

$$y = \log\left(\frac{\text{delivery\_days}}{\text{promised\_delivery\_days}}\right)$$

Predictions are converted back to days before computing metrics. The log also reduces the influence of the few very long deliveries (up to 209 days).

The baseline every model must beat: `promised_delivery_days` × the median train ratio of delivery days to promised days.

In [7]:
# Delivery time drifts down; the promise drifts with it
display(
    df.groupby("split", sort=False)[[target, "promised_delivery_days"]]
    .mean()
    .round(2)
)


def to_log_ratio(days, promised):
    return np.log(days / promised)


def to_days(log_ratio, promised):
    return np.exp(log_ratio) * promised


y_train_ratio = to_log_ratio(y_train, X_train["promised_delivery_days"])

print("Log-ratio target (train):")
print(y_train_ratio.describe().round(3))

,delivery_days,promised_delivery_days
split,,
train,13.19,25.32
validation,8.96,20.33
test,7.73,15.82


Log-ratio target (train):
count    83947.000
mean        -0.818
std          0.579
min         -3.736
25%         -1.177
50%         -0.822
75%         -0.458
max          2.527
dtype: float64


In [8]:
tscv = TimeSeriesSplit(n_splits=CV_FOLDS)


def regression_report(name, y_true, y_pred):
    return {
        "Model": name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
        "Bias": np.mean(y_pred - y_true),  # > 0: predicts slower than actual
    }


def cv_mae(model, X):
    """MAE in days of `model` (fitted on the log-ratio target) over the time-ordered folds."""
    scores = []
    for fit_idx, eval_idx in tscv.split(X):
        fold_model = clone(model).fit(X.iloc[fit_idx], y_train_ratio.iloc[fit_idx])
        pred = to_days(
            fold_model.predict(X.iloc[eval_idx]),
            X_train["promised_delivery_days"].iloc[eval_idx],
        )
        scores.append(mean_absolute_error(y_train.iloc[eval_idx], pred))
    return np.mean(scores), np.std(scores)


class PromiseBaseline:
    """Predicts the median train log-ratio for every order, i.e. promised days x median ratio."""

    def fit(self, X, y):
        self.median_ = np.median(y)
        return self

    def predict(self, X):
        return np.full(len(X), self.median_)

    def get_params(self, deep=True):
        return {}

    def set_params(self, **params):
        return self


baseline = PromiseBaseline().fit(X_train, y_train_ratio)
baseline_val = to_days(baseline.predict(X_val), X_val["promised_delivery_days"])

print("Median ratio delivery/promised:", round(np.exp(baseline.median_), 3))
display(pd.DataFrame([regression_report("Baseline: promised x ratio", y_val, baseline_val)]).round(3))

Median ratio delivery/promised: 0.439


,Model,MAE,RMSE,R2,Bias
0,Baseline: promised x ratio,3.58,5.598,0.185,-0.026


## 4. LassoCV

### 4.1 Lasso preprocessing

In [9]:
# divide features into numeric and categorical
# X_train already has the correlation filter applied
numeric_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    exclude=["number"]
).columns.tolist()

print("\nNumeric count:", len(numeric_features))
print("Categorical count:", len(categorical_features))

# numeric features
numeric_transformer = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    )
])

# categorical features
categorical_transformer = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])

# define preprocessor
preprocessor = ColumnTransformer([
    (
        "num",
        numeric_transformer,
        numeric_features
    ),
    (
        "cat",
        categorical_transformer,
        categorical_features
    )
])


Numeric count: 45
Categorical count: 7


### 4.2 LassoCV with time-ordered folds

`best_alpha` minimises the mean CV error over the time-ordered folds.

The one-standard-error alpha (largest alpha within one standard error of the best) was also tried for selection. It kept only 16 features and lost to the baseline on validation (MAE 3.76 vs 3.58), so selection uses `best_alpha`.

In [10]:
lasso_pipeline = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "lasso",
        LassoCV(
            cv=tscv,
            max_iter=20000
        )
    )
])

lasso_pipeline.fit(
    X_train,
    y_train_ratio
)

lasso_cv = lasso_pipeline.named_steps["lasso"]

best_alpha = lasso_cv.alpha_

print("Best alpha:", best_alpha)

Best alpha: 0.004061171534340354


In [11]:
# check Lasso coefficients (at best alpha, fitted on the full training set)
feature_names = (
    lasso_pipeline
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

lasso_coef = lasso_cv.coef_

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": lasso_coef
})

coef_df["abs_coefficient"] = (
    coef_df["coefficient"].abs()
)

coef_df = coef_df.sort_values(
    "abs_coefficient",
    ascending=False
)

print("Coefficients are on the log-ratio scale: +0.1 is about 10% slower than the promise ratio.")
print("Non-zero at best alpha:", (lasso_coef != 0).sum(), "of", len(lasso_coef))
display(coef_df.head(15))

Coefficients are on the log-ratio scale: +0.1 is about 10% slower than the promise ratio.
Non-zero at best alpha: 42 of 204


,feature,coefficient,abs_coefficient
27,num__promised_delivery_days,-0.183412,0.183412
12,num__any_seller_same_state,-0.121034,0.121034
39,num__seller_past_delivery_days,0.118310,0.118310
16,num__estimated_max_distance_km,0.118290,0.118290
70,cat__customer_state_SP,-0.095213,0.095213
170,cat__primary_seller_state_SP,0.078520,0.078520
31,num__purchase_month_cos,0.074919,0.074919
200,cat__payment_type_boleto,0.074822,0.074822
55,cat__customer_state_MG,-0.053366,0.053366
175,cat__primary_seller_region_SE,0.052428,0.052428


## 5. Stability selection

Refit Lasso at `best_alpha` on the training window of each time-ordered fold. A categorical feature counts as selected in a fold if any of its one-hot columns is non-zero. Keep features selected in at least 80% of folds.

In [12]:
def encoded_to_original(fitted_preprocessor):
    """Map each encoded column name (num__x, cat__x_value) to its original feature."""
    feature_map = {f"num__{col}": col for col in numeric_features}

    ohe = (
        fitted_preprocessor
        .named_transformers_["cat"]
        .named_steps["onehot"]
    )
    for original_col, categories in zip(categorical_features, ohe.categories_):
        for category in categories:
            feature_map[f"cat__{original_col}_{category}"] = original_col
    return feature_map


STABILITY_THRESHOLD = 0.8

selection_counts = pd.Series(0, index=feature_cols_corr)

for fit_idx, _ in tscv.split(X_train):
    fold_model = Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("lasso", Lasso(alpha=best_alpha, max_iter=20000)),
    ]).fit(X_train.iloc[fit_idx], y_train_ratio.iloc[fit_idx])

    names = fold_model.named_steps["preprocessor"].get_feature_names_out()
    feature_map = encoded_to_original(fold_model.named_steps["preprocessor"])
    nonzero = names[fold_model.named_steps["lasso"].coef_ != 0]
    used = {feature_map[name] for name in nonzero}
    selection_counts[list(used)] += 1

selection_freq = (selection_counts / tscv.get_n_splits()).sort_values(ascending=False)

final_features = selection_freq[selection_freq >= STABILITY_THRESHOLD].index.tolist()

assert not set(final_features) & set(corr_manual_drop)

print("Selected in every fold:", int((selection_freq == 1).sum()))
print("Selected in some folds:", int(((selection_freq > 0) & (selection_freq < 1)).sum()))
print("Never selected:", int((selection_freq == 0).sum()))
print(f"\nNumber of final original features (>= {STABILITY_THRESHOLD:.0%} of folds):", len(final_features))
display(selection_freq.to_frame("selection_frequency"))

Selected in every fold: 26
Selected in some folds: 22
Never selected: 4

Number of final original features (>= 80% of folds): 33


,selection_frequency
customer_state,1.0
any_seller_same_city,1.0
mean_seller_zip_lng,1.0
estimated_max_distance_km,1.0
geolocation_missing,1.0
longest_side_cm,1.0
payment_type,1.0
payment_missing,1.0
promised_delivery_days,1.0
promised_days_per_km,1.0


### Compare models

All models predict the log-ratio target and are scored in days. `CV MAE` is the mean ± std over the time-ordered folds of the training set; validation is July 2018 only, which is unusually fast, so weigh both. The test set is not used here.

In [13]:
def lasso_on(columns, alpha):
    numeric = [c for c in numeric_features if c in columns]
    categorical = [c for c in categorical_features if c in columns]
    return Pipeline([
        ("preprocessor", ColumnTransformer([
            ("num", clone(numeric_transformer), numeric),
            ("cat", clone(categorical_transformer), categorical),
        ])),
        ("model", Lasso(alpha=alpha, max_iter=20000)),
    ])


full_model = Pipeline([
    (
        "preprocessor",
        clone(preprocessor)
    ),
    (
        "model",
        LinearRegression()
    )
])

candidates = {
    "Baseline: promised x ratio": (baseline, feature_cols_corr),
    f"Linear Regression, {len(feature_cols_corr)} features": (full_model, feature_cols_corr),
    f"Lasso best alpha, {len(feature_cols_corr)} features": (lasso_on(feature_cols_corr, best_alpha), feature_cols_corr),
    f"Lasso best alpha, {len(final_features)} selected features": (lasso_on(final_features, best_alpha), final_features),
}

rows = []
for name, (model, columns) in candidates.items():
    cv_mean, cv_std = cv_mae(model, X_train[columns])
    fitted = clone(model).fit(X_train[columns], y_train_ratio)
    val_pred = to_days(fitted.predict(X_val[columns]), X_val["promised_delivery_days"])
    rows.append({
        **regression_report(name, y_val, val_pred),
        "CV MAE": f"{cv_mean:.3f} ± {cv_std:.3f}",
    })

comparison = pd.DataFrame(rows).set_index("Model")
display(comparison.round(3))

,MAE,RMSE,R2,Bias,CV MAE
Model,,,,,
Baseline: promised x ratio,3.580,5.598,0.185,-0.026,5.755 ± 0.952
"Linear Regression, 52 features",3.237,5.231,0.288,-0.356,5.517 ± 1.006
"Lasso best alpha, 52 features",3.285,5.248,0.284,-0.163,5.303 ± 0.762
"Lasso best alpha, 33 selected features",3.361,5.282,0.274,0.092,5.302 ± 0.752


In [14]:
baseline_mae = comparison.loc["Baseline: promised x ratio", "MAE"]
selected_mae = comparison.iloc[-1]["MAE"]
print(f"Selected Lasso vs baseline on validation: {baseline_mae - selected_mae:+.3f} days MAE improvement")

Selected Lasso vs baseline on validation: +0.218 days MAE improvement


## 6. Extract selected features

In [15]:
X_train_final = train_df[
    final_features
].copy()

y_train_final = train_df[
    "delivery_days"
].copy()


X_val_final = val_df[
    final_features
].copy()

y_val_final = val_df[
    "delivery_days"
].copy()


X_test_final = test_df[
    final_features
].copy()

y_test_final = test_df[
    "delivery_days"
].copy()

print("Train X:", X_train_final.shape)
print("Validation X:", X_val_final.shape)
print("Test X:", X_test_final.shape)

Train X: (83947, 33)
Validation X: (6156, 33)
Test X: (6351, 33)


In [16]:
output_path = TABLES_DIR / "regression_selected_features.csv"

final_feature_df = (
    selection_freq
    .rename("selection_frequency")
    .rename_axis("feature")
    .reset_index()
)
final_feature_df["selected"] = final_feature_df["feature"].isin(final_features)

final_feature_df.to_csv(
    output_path,
    index=False
)

print("Saved to:", output_path.relative_to(ROOT))

Saved to: report/tables/regression_selected_features.csv
